<a href="https://colab.research.google.com/github/MohamedAamer01/otto-recommender-sys/blob/main/notebooks/01_Data_Understanding_OTTO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
BASE_DIR = "/content/drive/MyDrive/OTTO_Recommender_System"

folders = [
    "data/raw",
    "data/processed",
    "notebooks",
    "results"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("Structure du projet créée avec succès !")


Structure du projet créée avec succès !


In [ ]:
!pip install -q --upgrade kaggle

In [ ]:
from google.colab import userdata
import os

token = userdata.get("KAGGLE_API_TOKEN")

assert token, "Token introuvable"

os.environ["KAGGLE_API_TOKEN"] = token

print("Kaggle API Token configuré avec succès !")

Kaggle API Token configuré avec succès !


In [ ]:
!kaggle competitions list -v

ref,deadline,category,reward,teamCount,userHasEntered,userRank
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge,2017-12-15 23:59:00,Featured,"1,500,000 Usd",518,False,0
https://www.kaggle.com/competitions/zillow-prize-1,2018-01-10 15:59:00,Featured,"1,200,000 Usd",3770,False,0
https://www.kaggle.com/competitions/data-science-bowl-2017,2017-04-12 23:59:00,Featured,"1,000,000 Usd",1972,False,0
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection,2023-06-14 23:59:00,Featured,"1,000,000 Usd",1249,False,0
https://www.kaggle.com/competitions/gemini-3,2025-12-12 23:59:00,Featured,"500,000 Usd",4079,False,0
https://www.kaggle.com/competitions/openai-gpt-oss-20b-red-teaming,2025-08-26 23:59:00,Featured,"500,000 Usd",601,False,0
https://www.kaggle.com/competitions/openai-to-z-challenge,2025-06-29 23:59:00,Featured,"500,000 Usd",225,False,0
https://www.kaggle.com/competitions/arc-prize-2026-paper-track,2026-11-09 23:59:00,Featured,"450,000 Usd",679,False

In [ ]:
#Telechargement
import os

DATA_DIR = "/content/otto_data"
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle competitions download \
    -c otto-recommender-system \
    -p /content/otto_data

100% 1.94G/1.94G [00:17<00:00, 120MB/s]



Ectraire les fichiers

In [ ]:
import zipfile
import os

DATA_DIR = "/content/otto_data"
zip_path = os.path.join(DATA_DIR, "otto-recommender-system.zip")
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(DATA_DIR)

print("Extraction terminée !")
print(os.listdir(DATA_DIR))

Extraction terminée !
['train.jsonl', 'sample_submission.csv', 'otto-recommender-system.zip', 'test.jsonl']


Verifier les fichiers

In [ ]:
for file in os.listdir(DATA_DIR):
    path = os.path.join(DATA_DIR, file)
    if os.path.isfile(path):
        print(f"{file} : {os.path.getsize(path) / (1024**3):.2f} Go")

train.jsonl : 10.53 Go
sample_submission.csv : 0.17 Go
otto-recommender-system.zip : 1.94 Go
test.jsonl : 0.37 Go


Examiner qlqs sessions

In [ ]:
import json
train_path = os.path.join(DATA_DIR, "train.jsonl")
with open(train_path, "r") as f:
    for i in range(3):
        session = json.loads(next(f))
        print(f"\nSession ID : {session['session']}")
        print(f"Nombre d'événements : {len(session['events'])}")
        print("Premiers événements :", session["events"][:3])


Session ID : 0
Nombre d'événements : 276
Premiers événements : [{'aid': 1517085, 'ts': 1659304800025, 'type': 'clicks'}, {'aid': 1563459, 'ts': 1659304904511, 'type': 'clicks'}, {'aid': 1309446, 'ts': 1659367439426, 'type': 'clicks'}]

Session ID : 1
Nombre d'événements : 32
Premiers événements : [{'aid': 424964, 'ts': 1659304800025, 'type': 'carts'}, {'aid': 1492293, 'ts': 1659304852871, 'type': 'clicks'}, {'aid': 1492293, 'ts': 1659304863627, 'type': 'carts'}]

Session ID : 2
Nombre d'événements : 33
Premiers événements : [{'aid': 763743, 'ts': 1659304800038, 'type': 'clicks'}, {'aid': 137492, 'ts': 1659304866606, 'type': 'clicks'}, {'aid': 504789, 'ts': 1659304915207, 'type': 'clicks'}]


Pour analyser, on veut un DataFrame plat : une ligne = un événement

In [ ]:
import pandas as pd
import numpy as np
from itertools import islice

rows = []
with open(train_path, "r") as f:
    for line in islice(f, 1000):
        rec = json.loads(line)
        sid = rec["session"]
        for ev in rec["events"]:
            rows.append((sid, ev["aid"], ev["ts"], ev["type"]))

df_sample = pd.DataFrame(rows, columns=["session", "aid", "ts", "type"])

# Types compacts : int32 suffit pour session et aid
df_sample["session"] = df_sample["session"].astype("int32")
df_sample["aid"] = df_sample["aid"].astype("int32")
df_sample["type"] = df_sample["type"].astype("category")

print("Shape :", df_sample.shape)
print(df_sample.dtypes)
print(f"Mémoire : {df_sample.memory_usage(deep=True).sum() / 1e6:.2f} MB")
display(df_sample.head(10))

Shape : (80267, 4)
session       int32
aid           int32
ts            int64
type       category
dtype: object
Mémoire : 1.36 MB


,session,aid,ts,type
0,0,1517085,1659304800025,clicks
1,0,1563459,1659304904511,clicks
2,0,1309446,1659367439426,clicks
3,0,16246,1659367719997,clicks
4,0,1781822,1659367871344,clicks
5,0,1152674,1659367885796,clicks
6,0,1649869,1659369893840,carts
7,0,461689,1659369898050,carts
8,0,305831,1659370027105,orders
9,0,461689,1659370027105,orders


In [ ]:
#Verification
# 1. Valeurs manquantes
print(df_sample.isna().sum())

# 2. Répartition des types
print(df_sample["type"].value_counts(normalize=True))

# 3. Conversion de ts en date lisible
df_sample["datetime"] = pd.to_datetime(df_sample["ts"], unit="ms")
print(df_sample["datetime"].min(), "->", df_sample["datetime"].max())

# 4. Les événements sont-ils triés par ts dans chaque session ?
is_sorted = df_sample.groupby("session")["ts"].apply(lambda s: s.is_monotonic_increasing)
print("Sessions triées par ts :", is_sorted.mean() * 100, "%")

# 5. Taille des sessions
sizes = df_sample.groupby("session").size()
print(sizes.describe())

session    0
aid        0
ts         0
type       0
dtype: int64
type
clicks    0.896271
carts     0.081590
orders    0.022139
Name: proportion, dtype: float64
2022-07-31 22:00:00.025000 -> 2022-08-28 21:59:51.257000
Sessions triées par ts : 100.0 %
count    1000.00000
mean       80.26700
std        94.87775
min         2.00000
25%        11.00000
50%        41.00000
75%       115.00000
max       482.00000
dtype: float64


Conptage et Streaning

In [ ]:
from collections import Counter
import time

n_sessions = 0
n_events = 0
type_counts = Counter()
unsorted_sessions = 0
ts_min, ts_max = float("inf"), 0
session_sizes = []          # un entier par session : ~13M entiers, acceptable

t0 = time.time()
with open(train_path, "r") as f:
    for line in f:
        rec = json.loads(line)
        evs = rec["events"]
        n_sessions += 1
        n_events += len(evs)
        session_sizes.append(len(evs))

        ts_list = [e["ts"] for e in evs]
        if ts_list != sorted(ts_list):
            unsorted_sessions += 1
        ts_min = min(ts_min, ts_list[0])
        ts_max = max(ts_max, ts_list[-1])

        for e in evs:
            type_counts[e["type"]] += 1

        if n_sessions % 2_000_000 == 0:
            print(f"{n_sessions:,} sessions traitées ({time.time()-t0:.0f}s)")

print(f"\nSessions : {n_sessions:,}")
print(f"Événements : {n_events:,}")
print(type_counts)
print(f"Sessions non triées : {unsorted_sessions}")
print(f"Durée : {time.time()-t0:.0f}s")

2,000,000 sessions traitées (124s)
4,000,000 sessions traitées (207s)
6,000,000 sessions traitées (273s)
8,000,000 sessions traitées (327s)
10,000,000 sessions traitées (374s)
12,000,000 sessions traitées (414s)

Sessions : 12,899,779
Événements : 216,716,096
Counter({'clicks': 194720954, 'carts': 16896191, 'orders': 5098951})
Sessions non triées : 0
Durée : 431s


In [ ]:
import json, time
from array import array
from collections import Counter

n_sessions = 0
n_events = 0
type_counts = Counter()
unsorted_sessions = 0
ts_min, ts_max = float("inf"), 0
session_sizes = array("i")          # tailles de sessions, compact

t0 = time.time()
with open(train_path, "r") as f:
    for line in f:
        rec = json.loads(line)
        evs = rec["events"]
        n = len(evs)

        n_sessions += 1
        n_events += n
        session_sizes.append(n)

        ts_list = [e["ts"] for e in evs]
        # Vérif tri : comparer chaque ts au suivant (plus léger que sorted())
        if any(ts_list[i] > ts_list[i+1] for i in range(n - 1)):
            unsorted_sessions += 1
        ts_min = min(ts_min, ts_list[0])
        ts_max = max(ts_max, ts_list[-1])

        type_counts.update(e["type"] for e in evs)

        if n_sessions % 1_000_000 == 0:
            print(f"{n_sessions:,} sessions ({time.time()-t0:.0f}s)")

print(f"\nSessions : {n_sessions:,}")
print(f"Événements : {n_events:,}")
print(dict(type_counts))
print(f"Sessions non triées : {unsorted_sessions:,}")
print(f"ts min/max : {ts_min} -> {ts_max}")
print(f"Durée : {(time.time()-t0)/60:.1f} min")

1,000,000 sessions (80s)
2,000,000 sessions (134s)
3,000,000 sessions (174s)
4,000,000 sessions (211s)
5,000,000 sessions (244s)
6,000,000 sessions (275s)
7,000,000 sessions (302s)
8,000,000 sessions (326s)
9,000,000 sessions (350s)
10,000,000 sessions (373s)
11,000,000 sessions (394s)
12,000,000 sessions (413s)

Sessions : 12,899,779
Événements : 216,716,096
{'clicks': 194720954, 'carts': 16896191, 'orders': 5098951}
Sessions non triées : 0
ts min/max : 1659304800025 -> 1661723999984
Durée : 7.2 min


In [ ]:
import numpy as np
sizes = np.frombuffer(session_sizes, dtype=np.int32)   # vue NumPy sans copie
print(f"Moyenne : {sizes.mean():.1f}")
print(f"Écart-type : {sizes.std():.1f}")
print("Quantiles :")
for q in [0, 25, 50, 75, 95, 99, 99.9, 100]:
    print(f"  {q:>5}% : {np.percentile(sizes, q):.0f}")
print(f"Sessions d'un seul événement : {(sizes == 1).sum():,}")

Moyenne : 16.8
Écart-type : 33.6
Quantiles :
      0% : 2
     25% : 3
     50% : 6
     75% : 15
     95% : 68
     99% : 176
   99.9% : 358
    100% : 500
Sessions d'un seul événement : 0



1.   Nombre d’articles (aid) uniques
2.   Valeurs manquantes sur tout le fichier

In [ ]:
from collections import Counter
import time

aid_counts = Counter()        # popularité globale de chaque article
missing_keys = 0
expected = {"aid", "ts", "type"}

t0 = time.time()
with open(train_path, "r") as f:
    for line in f:
        rec = json.loads(line)
        for e in rec["events"]:
            if e.keys() != expected:      # clé manquante ou en trop
                missing_keys += 1
            aid_counts[e["aid"]] += 1

print(f"Articles uniques : {len(aid_counts):,}")
print(f"Événements avec clés anormales : {missing_keys}")
print(f"Top 10 :", aid_counts.most_common(10))
print(f"Durée : {(time.time()-t0)/60:.1f} min")

Articles uniques : 1,855,603
Événements avec clés anormales : 0
Top 10 : [(1460571, 129004), (485256, 126836), (108125, 118524), (29735, 113279), (1733943, 105091), (832192, 91325), (184976, 90244), (166037, 84657), (554660, 80197), (231487, 79872)]
Durée : 9.6 min


In [ ]:
test_path = os.path.join(DATA_DIR, "test.jsonl")

n_sess, n_ev = 0, 0
sizes_test = array("i")
with open(test_path, "r") as f:
    for line in f:
        rec = json.loads(line)
        n_sess += 1
        n_ev += len(rec["events"])
        sizes_test.append(len(rec["events"]))

st = np.frombuffer(sizes_test, dtype=np.int32)
print(f"Sessions test : {n_sess:,} | Événements : {n_ev:,}")
print(f"Médiane : {np.median(st):.0f} | Moyenne : {st.mean():.1f} | Max : {st.max()}")

sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
print(sub.shape)
display(sub.head())

Sessions test : 1,671,803 | Événements : 6,928,123
Médiane : 2 | Moyenne : 4.1 | Max : 458
(5015409, 2)


,session_type,labels
0,12899779_clicks,129004 126836 118524
1,12899779_carts,129004 126836 118524
2,12899779_orders,129004 126836 118524
3,12899780_clicks,129004 126836 118524
4,12899780_carts,129004 126836 118524



# Conclusion de l'exploration de données (EDA)

L'analyse globale du jeu de données OTTO permet de poser les bases de notre stratégie :

* **Volumétrie et Qualité** : Le jeu de données est massif avec 12,9 millions de sessions d'entraînement (216,7 millions d'événements) réparties sur 4 semaines. L'intégrité est excellente : aucune valeur manquante et l'intégralité des sessions est triée par ordre chronologique.

* **Structure des Sessions** : La distribution des longueurs de sessions est asymétrique à droite, caractérisée par une médiane de 6 événements et une moyenne de 16,8. De plus, les sessions sont limitées artificiellement par une troncature stricte entre un minimum de 2 événements et un maximum de 500 événements.

* **Déséquilibre des Interactions** : Il existe une forte disparité dans les types d'événements : 90 % de clics, 8 % d'ajouts au panier et seulement 2 % d'achats. Cependant, l'optimisation devra se focaliser sur les achats car ils représentent 60 % du score final.

* **Leçon Méthodologique** : L'analyse initiale a confirmé que les premières lignes d'un fichier volumineux ne forment pas un échantillon représentatif de l'ensemble de la population.